#### You had 45 separate raw CSV files, one per pollution monitoring station, sitting messily in data/raw/cpcb/. Each file had a full year of daily pollution readings, but nothing linked them to your clean stations_master.csv — they were just floating files with confusing names. Notebook 2's job was to connect the dots: read all 45 files, figure out which station each one belongs to, summarize a year of daily data into simple stats, and save two clean, usable files.

#### What Actually Happened, Step by Step
1. Loaded the "directory" — stations_master.csv (your 45 clean station records) was loaded so the code would know what a "correct" station name looks like.

2. Decoded each filename — For all 45 raw files, the code stripped away junk text (raw_data_data_, _1D, "Delhi") to extract just the station name and network (DPCC/IITM/CPCB).

3. Matched each file to its station — Using that extracted name, it looked up the matching row in your master table, hit a snag with 3 files (Dwarka-Sector 8, IGNOU Maidan Garhi, Okhla Phase-2) due to punctuation mismatches, then fixed it with a normalizer function so all 45 matched correctly.

4. Read and cleaned each file — For every matched file, it loaded the daily data, dropped the 6 completely empty columns (Xylene, VWS, etc.), and tagged each row with its station_id.

5. Computed summary statistics — For every station, it calculated the yearly mean, median, max, and missing-data percentage for each pollutant (PM2.5, PM10, NO2, etc.) — turning 365 daily rows into one summary row per station.

6. Saved two final outputs — one summary table, one full combined daily table.

#### What Each Processed File Means
File	What's inside	Think of it as
station_pollution_stats.csv	45 rows — one per station, with average/median/max pollution levels and data-quality flags for the whole year	A "report card" for each station — e.g., "Anand Vihar: PM2.5 average = 146 µg/m³, 0.3% data missing"
station_daily_combined.csv	~16,000+ rows — every single daily reading from all 45 stations, stacked into one long table	The full "raw diary" — every day, every station, all in one place for future trend/seasonal analysis

##### Notebook 2 turned 45 disconnected, messy files into one clean "pollution scorecard" per station — and that scorecard is exactly what gets attached to each of your 2,425 schools in the next step.

### Imports & Paths

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

RAW_DIR = Path("../data/raw/cpcb")
MASTER_PATH = Path("../data/processed/stations_master.csv")
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

### Filename Parser

In [15]:
def parse_filename(fname: str):
    base = fname.replace(".csv", "")
    base = re.sub(r"^raw_data_data_", "", base, flags=re.IGNORECASE)
    base = re.sub(r"_+1d.*$", "", base, flags=re.IGNORECASE)
    base = base.strip("_").strip()
    m = re.search(r"[-_]\s*(dpcc|iitm|cpcb)\s*$", base, flags=re.IGNORECASE)
    network = m.group(1).upper() if m else None
    name_part = re.sub(
        r",?[-_]*\s*delhi\s*[-_]+\s*(dpcc|iitm|cpcb)\s*$",
        "", base, flags=re.IGNORECASE
    )
    name_part = name_part.replace("_", " ").replace("-", " ").strip()
    name_part = re.sub(r"\s+", " ", name_part)
    name_part = name_part.title()
    return name_part, network

def normalize_name(name: str) -> str:
    name = name.lower()
    name = re.sub(r"[,\-_]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name

### Load Station Master & Build Match Key

In [16]:
master = pd.read_csv(MASTER_PATH)

master["match_key"] = (
    master["station_name"].apply(normalize_name)
    .str.replace(r"\(.*?\)", "", regex=True).str.strip()
    + "|" + master["network"].str.lower()
)

print(master.shape)
master.head()

(45, 6)


,station_id,station_name,network,latitude,longitude,match_key
0,STN_001,Alipur,DPCC,28.815329,77.153010,alipur|dpcc
1,STN_002,Anand Vihar,DPCC,28.647622,77.315809,anand vihar|dpcc
2,STN_003,Ashok Vihar,DPCC,28.695381,77.181665,ashok vihar|dpcc
3,STN_004,Aya Nagar,IITM,28.470691,77.109936,aya nagar|iitm
4,STN_005,Bawana,DPCC,28.776200,77.051074,bawana|dpcc


### Constants

In [4]:
DROP_COLS = [
    "Xylene (µg/m³)", "O Xylene (µg/m³)", "Eth-Benzene (µg/m³)",
    "MP-Xylene (µg/m³)", "RF (mm)", "VWS (m/s)"
]

POLLUTANTS = [
    "PM2.5 (µg/m³)", "PM10 (µg/m³)", "NO (µg/m³)", "NO2 (µg/m³)",
    "NOx (ppb)", "NH3 (µg/m³)", "SO2 (µg/m³)", "CO (mg/m³)",
    "Ozone (µg/m³)", "Benzene (µg/m³)", "Toluene (µg/m³)"
]

### Main Ingestion Loop

For each of the 45 files (inside raw/cpcb), it figures out which station it belongs to, opens the file, throws away useless empty columns, and calculates the yearly average/max/missing-data-percentage for each pollutant — then stacks all 45 summaries into one list.

In [18]:
records = []
unmatched_files = []
all_daily = []

csv_files = sorted(RAW_DIR.glob("*.csv"))
print(f"Found {len(csv_files)} raw station files")

for fpath in csv_files:
    name_part, network = parse_filename(fpath.name)
    if network is None:
        unmatched_files.append(fpath.name)
        continue

    key = normalize_name(name_part) + "|" + network.lower()
    match = master[master["match_key"] == key]

    if match.empty:
        alt = master[master["station_name"].str.lower().str.contains(re.escape(name_part.lower()), na=False)]
        if len(alt) == 1:
            match = alt
        else:
            unmatched_files.append(fpath.name)
            continue

    station_id = match.iloc[0]["station_id"]
    station_name = match.iloc[0]["station_name"]

    df = pd.read_csv(fpath)
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
    df["station_id"] = station_id
    df["station_name"] = station_name
    df["network"] = network
    df["source_file"] = fpath.name

    all_daily.append(df)

    stats = {
        "station_id": station_id, "station_name": station_name,
        "network": network, "n_days": len(df),
        "date_min": df["Timestamp"].min(), "date_max": df["Timestamp"].max()
    }

    for pol in POLLUTANTS:
        if pol in df.columns:
            stats[f"{pol}_mean"] = df[pol].mean()
            stats[f"{pol}_median"] = df[pol].median()
            stats[f"{pol}_max"] = df[pol].max()
            stats[f"{pol}_missing_pct"] = df[pol].isna().mean() * 100
        else:
            stats[f"{pol}_mean"] = np.nan
            stats[f"{pol}_missing_pct"] = 100.0

    records.append(stats)

print(f"Successfully processed: {len(records)} stations")
print(f"Unmatched files: {unmatched_files}")

Found 45 raw station files
Successfully processed: 45 stations
Unmatched files: []


/Users/kumarb/Documents/My Projects/delhi_school_air_pollution/.venv/lib/python3.9/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/kumarb/Documents/My Projects/delhi_school_air_pollution/.venv/lib/python3.9/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/kumarb/Documents/My Projects/delhi_school_air_pollution/.venv/lib/python3.9/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/kumarb/Documents/My Projects/delhi_school_air_pollution/.venv/lib/python3.9/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/kumarb/Documents/My Projects/delhi_school_air_pollution/.venv/lib/python3.9/site-

In [19]:
station_stats = pd.DataFrame(records)
daily_combined = pd.concat(all_daily, ignore_index=True) if all_daily else pd.DataFrame()

print(station_stats.shape)
print(daily_combined.shape)
station_stats.head()

(45, 50)
(16121, 23)


,station_id,station_name,network,n_days,date_min,date_max,PM2.5 (µg/m³)_mean,PM2.5 (µg/m³)_median,PM2.5 (µg/m³)_max,PM2.5 (µg/m³)_missing_pct,...,Ozone (µg/m³)_max,Ozone (µg/m³)_missing_pct,Benzene (µg/m³)_mean,Benzene (µg/m³)_median,Benzene (µg/m³)_max,Benzene (µg/m³)_missing_pct,Toluene (µg/m³)_mean,Toluene (µg/m³)_median,Toluene (µg/m³)_max,Toluene (µg/m³)_missing_pct
0,STN_001,Alipur,DPCC,365,2025-01-01,2025-12-31,99.947233,80.21,358.79,0.000000,...,202.61,0.547945,1.352959,0.58,52.05,0.00000,4.921589,1.71,62.62,0.00000
1,STN_002,Anand Vihar,DPCC,365,2025-01-01,2025-12-31,122.056556,77.62,506.50,0.547945,...,47.19,0.273973,3.303019,2.25,40.40,1.09589,32.011939,27.44,109.86,1.09589
2,STN_003,Ashok Vihar,DPCC,365,2025-01-01,2025-12-31,112.189260,78.25,572.19,0.000000,...,160.44,0.000000,1.475178,0.49,13.71,0.00000,7.818932,2.61,70.04,0.00000
3,STN_004,Aya Nagar,IITM,365,2025-01-01,2025-12-31,76.782384,64.27,286.70,0.000000,...,195.24,0.000000,NaN,NaN,NaN,100.00000,NaN,NaN,NaN,100.00000
4,STN_005,Bawana,DPCC,365,2025-01-01,2025-12-31,123.343562,91.12,488.32,0.000000,...,115.10,0.000000,3.075671,1.37,208.04,0.00000,20.804329,11.96,208.10,0.00000


In [20]:
print("--- VALIDATION ---")
print("Duplicate station_ids in stats:", station_stats["station_id"].duplicated().sum())

missing_from_stats = set(master["station_id"]) - set(station_stats["station_id"])
print("Stations in master but missing from stats:", missing_from_stats)

extra_in_stats = set(station_stats["station_id"]) - set(master["station_id"])
print("Stations in stats but not in master:", extra_in_stats)

print("\nStations with >20% missing PM2.5 data (low reliability flag):")
print(station_stats[station_stats["PM2.5 (µg/m³)_missing_pct"] > 20][
    ["station_name", "PM2.5 (µg/m³)_missing_pct"]
])

--- VALIDATION ---
Duplicate station_ids in stats: 1
Stations in master but missing from stats: {'STN_008'}
Stations in stats but not in master: set()

Stations with >20% missing PM2.5 data (low reliability flag):
                   station_name  PM2.5 (µg/m³)_missing_pct
7   Commonwealth Sports Complex                 100.000000
13           IGNOU_Maidan Garhi                 100.000000
15                    IIT Delhi                  49.180328
20                          JNU                 100.000000
30               NSUT Jaffarpur                 100.000000
42             Talkatora Garden                 100.000000


In [21]:
station_stats.to_csv(OUT_DIR / "station_pollution_stats.csv", index=False)
daily_combined.to_csv(OUT_DIR / "station_daily_combined.csv", index=False)

print("Saved:")
print(OUT_DIR / "station_pollution_stats.csv")
print(OUT_DIR / "station_daily_combined.csv")

Saved:
../data/processed/station_pollution_stats.csv
../data/processed/station_daily_combined.csv
